In [1]:
import sys
from pathlib import Path

current = Path.cwd()

PROJECT_ROOT = None

for path in [current] + list(current.parents):
    if (path / "src").is_dir():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing 'src'."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

print("Project root:", PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

from src.datasets.ecg_dataset import (
    PTBXLDataset as PTBXLTestDataset,
)

from src.models.ecg.xresnet1d import (
    XResNet1D,
)

from src.evaluation.evaluator import (
    collect_ecg_predictions,
)

from src.evaluation.metrics import (
    multilabel_metrics,
    binary_confusion_matrix,
)

from src.utils.device import get_device
from src.utils.seed import set_seed

print("Imports successful.")

Project root: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service
Imports successful.


In [2]:
DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = (
    DATA_DIR / "processed"
)

ECG_PROCESSED_DIR = (
    PROCESSED_DIR
    / "ecg"
    / "ptbxl"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "ecg"
)

RESULTS_DIR = (
    PROCESSED_DIR
    / "evaluation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DEVICE = get_device()

set_seed(42)

print("Device:", DEVICE)
print("PTB-XL processed:", ECG_PROCESSED_DIR)
print("ECG checkpoints:", CHECKPOINT_DIR)

Device: cuda
PTB-XL processed: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl
ECG checkpoints: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\checkpoints\ecg


In [3]:
manifest_candidates = [
    ECG_PROCESSED_DIR / "ptbxl_manifest.csv",
    ECG_PROCESSED_DIR / "manifest.csv",
    ECG_PROCESSED_DIR / "ptbxl_processed.csv",
]

MANIFEST_PATH = next(
    (
        path
        for path in manifest_candidates
        if path.exists()
    ),
    None,
)

if MANIFEST_PATH is None:
    raise FileNotFoundError(
        "PTB-XL manifest was not found in "
        f"{ECG_PROCESSED_DIR}."
    )

print("Manifest:", MANIFEST_PATH)

Manifest: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\ptbxl_manifest.csv


In [4]:
manifest = pd.read_csv(
    MANIFEST_PATH
)

print("Shape:", manifest.shape)
print()
print(
    manifest.head()
)
print()
print(
    manifest.columns.tolist()
)

Shape: (21837, 20)

   record_id  patient_id                record_name  \
0          1     15709.0  records100/00000/00001_lr   
1          2     13243.0  records100/00000/00002_lr   
2          3     20372.0  records100/00000/00003_lr   
3          4     17014.0  records100/00000/00004_lr   
4          5     17448.0  records100/00000/00005_lr   

                                    processed_path  split  original_fs  NORM  \
0  data\processed\ecg\ptbxl\waveforms\00001_lr.npy   test          100   1.0   
1  data\processed\ecg\ptbxl\waveforms\00002_lr.npy  train          100   1.0   
2  data\processed\ecg\ptbxl\waveforms\00003_lr.npy  train          100   1.0   
3  data\processed\ecg\ptbxl\waveforms\00004_lr.npy   test          100   1.0   
4  data\processed\ecg\ptbxl\waveforms\00005_lr.npy  train          100   1.0   

    MI  STTC   CD  HYP  RHYTHM_SR  RHYTHM_AFIB  RHYTHM_AFLT  RHYTHM_STACH  \
0  0.0   0.0  0.0  0.0        1.0          0.0          0.0           0.0   
1  0.0   0.0  

In [5]:
DIAGNOSTIC_COLUMNS = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP",
]

In [6]:
rhythm_candidates = [
    col
    for col in manifest.columns
    if (
        "rhythm" in col.lower()
        or col.upper() in [
            "AFIB",
            "AFLT",
            "SBRAD",
            "SR",
        ]
    )
]

print(
    "Possible rhythm columns:"
)

print(rhythm_candidates)

Possible rhythm columns:
['RHYTHM_SR', 'RHYTHM_AFIB', 'RHYTHM_AFLT', 'RHYTHM_STACH', 'RHYTHM_SBRAD', 'RHYTHM_SARRH', 'RHYTHM_PSVT', 'RHYTHM_BIGU', 'RHYTHM_PACE']


In [7]:
RHYTHM_COLUMNS = [
    col
    for col in manifest.columns
    if col.upper().startswith("RHYTHM_")
]

if not RHYTHM_COLUMNS:
    raise ValueError(
        "No rhythm label columns were found in the PTB-XL manifest."
    )

print(
    "Using rhythm columns:",
    RHYTHM_COLUMNS,
)

Using rhythm columns: ['RHYTHM_SR', 'RHYTHM_AFIB', 'RHYTHM_AFLT', 'RHYTHM_STACH', 'RHYTHM_SBRAD', 'RHYTHM_SARRH', 'RHYTHM_PSVT', 'RHYTHM_BIGU', 'RHYTHM_PACE']


In [8]:
all_label_columns = (
    DIAGNOSTIC_COLUMNS
    + RHYTHM_COLUMNS
)

missing = [
    col
    for col in all_label_columns
    if col not in manifest.columns
]

if missing:
    raise ValueError(
        f"Missing label columns: {missing}"
    )

print(
    "Diagnostic labels:",
    DIAGNOSTIC_COLUMNS,
)

print(
    "Rhythm labels:",
    RHYTHM_COLUMNS,
)

Diagnostic labels: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
Rhythm labels: ['RHYTHM_SR', 'RHYTHM_AFIB', 'RHYTHM_AFLT', 'RHYTHM_STACH', 'RHYTHM_SBRAD', 'RHYTHM_SARRH', 'RHYTHM_PSVT', 'RHYTHM_BIGU', 'RHYTHM_PACE']


In [9]:
if "split" in manifest.columns:
    print(
        manifest["split"]
        .value_counts(
            dropna=False
        )
    )

test_count = (
    manifest["split"]
    .astype(str)
    .str.lower()
    .eq("test")
    .sum()
)

print(
    "\nTest records:",
    int(test_count),
)

split
train    15298
test      3273
val       3266
Name: count, dtype: int64

Test records: 3273


In [10]:
test_manifest = manifest[
    manifest["split"]
    .astype(str)
    .str.lower()
    == "test"
].copy()

test_manifest = (
    test_manifest
    .reset_index(drop=True)
)

def resolve_processed_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

resolved_paths = test_manifest["processed_path"].map(
    resolve_processed_path
)
exists_mask = resolved_paths.map(Path.exists)

test_manifest["resolved_path"] = resolved_paths

print(
    "Existing signals:",
    int(exists_mask.sum()),
)

print(
    "Missing signals:",
    int((~exists_mask).sum()),
)

if not exists_mask.all():
    print(
        test_manifest.loc[
            ~exists_mask,
            ["processed_path"]
        ].head(10)
    )

test_manifest = (
    test_manifest[exists_mask]
    .reset_index(drop=True)
)

print(
    "Usable test records:",
    len(test_manifest)
)

Existing signals: 3273
Missing signals: 0
Usable test records: 3273


In [11]:
test_signals = np.stack(
    [
        np.load(path)
        for path in test_manifest["resolved_path"]
    ]
)

record_paths = test_manifest[
    "processed_path"
].astype(str).tolist()

test_dataset = PTBXLTestDataset(
    signals=test_signals,
    labels_df=test_manifest,
    superclass_cols=DIAGNOSTIC_COLUMNS,
    rhythm_cols=RHYTHM_COLUMNS,
)

print(
    "Test signal shape:",
    test_signals.shape,
)

print(
    "Test dataset size:",
    len(test_dataset)
)

Test signal shape: (3273, 1000, 12)
Test dataset size: 3273


In [12]:
sample = test_dataset[0]

print(
    "Signal shape:",
    sample["signal"].shape,
)

print(
    "Diagnostic labels:",
    sample["diagnostic"]
)

print(
    "Rhythm labels:",
    sample["rhythm"]
)

Signal shape: torch.Size([12, 1000])
Diagnostic labels: tensor([1., 0., 0., 0., 0.])
Rhythm labels: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0.])


In [13]:
TEST_BATCH_SIZE = 32

test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print(
    "Test batches:",
    len(test_loader)
)

Test batches: 103


In [14]:
model = XResNet1D(
    input_channels=12,
    num_diagnostic_classes=len(
        DIAGNOSTIC_COLUMNS
    ),
    num_rhythm_classes=len(
        RHYTHM_COLUMNS
    ),
).to(DEVICE)

print(
    "Model created."
)

print(
    "Diagnostic outputs:",
    len(DIAGNOSTIC_COLUMNS)
)

print(
    "Rhythm outputs:",
    len(RHYTHM_COLUMNS)
)

Model created.
Diagnostic outputs: 5
Rhythm outputs: 9


In [15]:
checkpoint_candidates = [
    CHECKPOINT_DIR / "best_model.pt",
    CHECKPOINT_DIR / "ptbxl_best.pt",
    CHECKPOINT_DIR / "xresnet1d_best.pt",
    CHECKPOINT_DIR / "model.pt",
]

CHECKPOINT_PATH = None

for path in checkpoint_candidates:
    if path.exists():
        CHECKPOINT_PATH = path
        break

if CHECKPOINT_PATH is None:
    discovered = list(
        CHECKPOINT_DIR.glob("*.pt")
    )

    if discovered:
        CHECKPOINT_PATH = discovered[0]

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        f"No PTB-XL checkpoint found in "
        f"{CHECKPOINT_DIR}"
    )

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

Checkpoint: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\checkpoints\ecg\ptbxl_xresnet1d101.pt


In [16]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint[
            "state_dict"
        ]

    else:
        state_dict = checkpoint
else:
    state_dict = checkpoint

clean_state_dict = {}

for key, value in state_dict.items():
    new_key = key

    if new_key.startswith("module."):
        new_key = new_key[
            len("module.") :
        ]

    clean_state_dict[
        new_key
    ] = value

missing_keys, unexpected_keys = (
    model.load_state_dict(
        clean_state_dict,
        strict=False,
    )
)

print(
    "Missing keys:",
    missing_keys,
)

print(
    "Unexpected keys:",
    unexpected_keys,
)

if (
    len(missing_keys) == 0
    and len(unexpected_keys) == 0
):
    print(
        "Checkpoint loaded successfully."
    )

Missing keys: []
Unexpected keys: []
Checkpoint loaded successfully.


In [17]:
class PathAwareDataset:
    def __init__(self, dataset, paths):
        self.dataset = dataset
        self.paths = paths

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = dict(self.dataset[index])
        sample["record_path"] = self.paths[index]
        return sample


path_aware_dataset = PathAwareDataset(
    test_dataset,
    record_paths,
)

path_aware_loader = DataLoader(
    path_aware_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

predictions = collect_ecg_predictions(
    model=model,
    loader=path_aware_loader,
    device=DEVICE,
)

diagnostic_true = (
    predictions["diagnostic_true"]
)

diagnostic_prob = (
    predictions["diagnostic_prob"]
)

rhythm_true = (
    predictions["rhythm_true"]
)

rhythm_prob = (
    predictions["rhythm_prob"]
)

print(
    "Diagnostic shape:",
    diagnostic_true.shape,
)

print(
    "Rhythm shape:",
    rhythm_true.shape,
)

Diagnostic shape: (3273, 5)
Rhythm shape: (3273, 9)


In [18]:
diagnostic_metrics = (
    multilabel_metrics(
        diagnostic_true,
        diagnostic_prob,
    )
)

print(
    "Diagnostic macro AUROC:",
    diagnostic_metrics[
        "macro_AUROC"
    ],
)

print(
    "Diagnostic macro AUPRC:",
    diagnostic_metrics[
        "macro_AUPRC"
    ],
)

print(
    "Diagnostic macro F1:",
    diagnostic_metrics[
        "macro_F1"
    ],
)

Diagnostic macro AUROC: 0.8880132980751132
Diagnostic macro AUPRC: 0.7257649162357369
Diagnostic macro F1: 0.6701378399689993


In [19]:
diagnostic_auc_df = pd.DataFrame({
    "label": DIAGNOSTIC_COLUMNS,
    "AUROC": diagnostic_metrics[
        "per_label_AUROC"
    ],
})

diagnostic_auc_df

,label,AUROC
0,NORM,0.938506
1,MI,0.892374
2,STTC,0.908298
3,CD,0.912069
4,HYP,0.788820


In [20]:
rhythm_metrics = (
    multilabel_metrics(
        rhythm_true,
        rhythm_prob,
    )
)

print(
    "Rhythm macro AUROC:",
    rhythm_metrics[
        "macro_AUROC"
    ],
)

print(
    "Rhythm macro AUPRC:",
    rhythm_metrics[
        "macro_AUPRC"
    ],
)

print(
    "Rhythm macro F1:",
    rhythm_metrics[
        "macro_F1"
    ],
)

Rhythm macro AUROC: 0.9559134342946725
Rhythm macro AUPRC: 0.5701252538813704
Rhythm macro F1: 0.4616466519925905


In [21]:
rhythm_auc_df = pd.DataFrame({
    "label": RHYTHM_COLUMNS,
    "AUROC": rhythm_metrics[
        "per_label_AUROC"
    ],
})

rhythm_auc_df

,label,AUROC
0,RHYTHM_SR,0.920178
1,RHYTHM_AFIB,0.972954
2,RHYTHM_AFLT,0.933506
3,RHYTHM_STACH,0.994306
4,RHYTHM_SBRAD,0.979197
5,RHYTHM_SARRH,0.855666
6,RHYTHM_PSVT,0.992917
7,RHYTHM_BIGU,0.976757
8,RHYTHM_PACE,0.977739


In [22]:
summary_df = pd.DataFrame([
    {
        "task": "Diagnostic",
        "macro_AUROC":
            diagnostic_metrics[
                "macro_AUROC"
            ],
        "macro_AUPRC":
            diagnostic_metrics[
                "macro_AUPRC"
            ],
        "macro_F1":
            diagnostic_metrics[
                "macro_F1"
            ],
    },
    {
        "task": "Rhythm",
        "macro_AUROC":
            rhythm_metrics[
                "macro_AUROC"
            ],
        "macro_AUPRC":
            rhythm_metrics[
                "macro_AUPRC"
            ],
        "macro_F1":
            rhythm_metrics[
                "macro_F1"
            ],
    },
])

summary_df

,task,macro_AUROC,macro_AUPRC,macro_F1
0,Diagnostic,0.888013,0.725765,0.670138
1,Rhythm,0.955913,0.570125,0.461647


In [23]:
mi_index = (
    DIAGNOSTIC_COLUMNS.index("MI")
)

mi_true = diagnostic_true[
    :, mi_index
]

mi_prob = diagnostic_prob[
    :, mi_index
]

mi_cm = binary_confusion_matrix(
    mi_true,
    mi_prob,
)

mi_cm

array([[1870,  579],
       [ 111,  713]])

In [24]:
diagnostic_auc_df.sort_values(
    "AUROC",
    ascending=False,
)

,label,AUROC
0,NORM,0.938506
3,CD,0.912069
2,STTC,0.908298
1,MI,0.892374
4,HYP,0.788820


In [25]:
rhythm_auc_df.sort_values(
    "AUROC",
    ascending=False,
)

,label,AUROC
3,RHYTHM_STACH,0.994306
6,RHYTHM_PSVT,0.992917
4,RHYTHM_SBRAD,0.979197
8,RHYTHM_PACE,0.977739
7,RHYTHM_BIGU,0.976757
1,RHYTHM_AFIB,0.972954
2,RHYTHM_AFLT,0.933506
0,RHYTHM_SR,0.920178
5,RHYTHM_SARRH,0.855666


In [26]:
summary_path = (
    RESULTS_DIR
    / "ptbxl_test_metrics.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

print(
    "Saved:",
    summary_path
)

Saved: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\evaluation\ptbxl_test_metrics.csv


In [27]:
diagnostic_path = (
    RESULTS_DIR
    / "ptbxl_diagnostic_auc.csv"
)

rhythm_path = (
    RESULTS_DIR
    / "ptbxl_rhythm_auc.csv"
)

diagnostic_auc_df.to_csv(
    diagnostic_path,
    index=False,
)

rhythm_auc_df.to_csv(
    rhythm_path,
    index=False,
)

print(
    "Saved diagnostic:",
    diagnostic_path,
)

print(
    "Saved rhythm:",
    rhythm_path,
)

Saved diagnostic: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\evaluation\ptbxl_diagnostic_auc.csv
Saved rhythm: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\evaluation\ptbxl_rhythm_auc.csv


In [28]:
validation_manifest = manifest[
    manifest["split"].astype(str).str.lower() == "val"
].copy().reset_index(drop=True)
validation_manifest["resolved_path"] = validation_manifest[
    "processed_path"
].map(resolve_processed_path)
validation_manifest = validation_manifest[
    validation_manifest["resolved_path"].map(Path.exists)
].reset_index(drop=True)

validation_signals = np.stack([
    np.load(path)
    for path in validation_manifest["resolved_path"]
])
validation_dataset = PTBXLTestDataset(
    signals=validation_signals,
    labels_df=validation_manifest,
    superclass_cols=DIAGNOSTIC_COLUMNS,
    rhythm_cols=RHYTHM_COLUMNS,
)
validation_paths = validation_manifest[
    "processed_path"
].astype(str).tolist()
validation_loader = DataLoader(
    PathAwareDataset(validation_dataset, validation_paths),
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
validation_predictions = collect_ecg_predictions(
    model=model,
    loader=validation_loader,
    device=DEVICE,
)

validation_rows = []
for i, path in enumerate(validation_predictions["record_paths"]):
    row = {"record_path": path}
    for j, label in enumerate(DIAGNOSTIC_COLUMNS):
        row[f"{label}_true"] = validation_predictions[
            "diagnostic_true"
        ][i, j]
        row[f"{label}_prob"] = validation_predictions[
            "diagnostic_prob"
        ][i, j]
    validation_rows.append(row)

pd.DataFrame(validation_rows).to_csv(
    RESULTS_DIR / "ptbxl_val_predictions.csv",
    index=False,
)
print("Saved real PTB-XL validation predictions")

Saved real PTB-XL validation predictions


In [29]:
prediction_rows = []

for i, path in enumerate(
    predictions["record_paths"]
):
    row = {
        "record_path": path,
    }

    for j, label in enumerate(
        DIAGNOSTIC_COLUMNS
    ):
        row[
            f"{label}_true"
        ] = diagnostic_true[i, j]

        row[
            f"{label}_prob"
        ] = diagnostic_prob[i, j]

    for j, label in enumerate(
        RHYTHM_COLUMNS
    ):
        row[
            f"{label}_true"
        ] = rhythm_true[i, j]

        row[
            f"{label}_prob"
        ] = rhythm_prob[i, j]

    prediction_rows.append(row)

predictions_df = pd.DataFrame(
    prediction_rows
)

predictions_path = (
    RESULTS_DIR
    / "ptbxl_test_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False,
)

print(
    "Saved:",
    predictions_path
)

Saved: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\evaluation\ptbxl_test_predictions.csv


In [30]:
print("PTB-XL Test Evaluation")
print("=" * 40)

print(
    f"Test records: {len(diagnostic_true)}"
)

print()
print("Diagnostic")
print(
    f"  Macro AUROC: {diagnostic_metrics['macro_AUROC']:.4f}"
)
print(
    f"  Macro AUPRC: {diagnostic_metrics['macro_AUPRC']:.4f}"
)
print(
    f"  Macro F1:    {diagnostic_metrics['macro_F1']:.4f}"
)

print()
print("Rhythm")
print(
    f"  Macro AUROC: {rhythm_metrics['macro_AUROC']:.4f}"
)
print(
    f"  Macro AUPRC: {rhythm_metrics['macro_AUPRC']:.4f}"
)
print(
    f"  Macro F1:    {rhythm_metrics['macro_F1']:.4f}"
)

PTB-XL Test Evaluation
Test records: 3273

Diagnostic
  Macro AUROC: 0.8880
  Macro AUPRC: 0.7258
  Macro F1:    0.6701

Rhythm
  Macro AUROC: 0.9559
  Macro AUPRC: 0.5701
  Macro F1:    0.4616
